In [38]:
import random
import itertools
import random
import itertools
import json
import numpy as np
import torch

from collections import defaultdict

import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from transformers import BertTokenizer
from transformers import BertModel

In [ ]:
with open('data/reddit_comment_body_dec_2024.json', 'r') as f:
    data = json.load(f)
    
# Create a dictionary to store messages by author
author_messages = defaultdict(list)
for item in data:
    author_messages[item['author']].append(item['body'])

Create Trainining Data

In [ ]:
def generate_training_data(author_messages, n_authors=5, num_examples=2000):
    n_pairs = num_examples // n_authors
    
    eligible_authors = [auth for auth, msgs in author_messages.items() if len(msgs) >= 2]
    if len(eligible_authors) < n_authors:
        raise ValueError(f"Not enough authors with at least 2 messages (found {len(eligible_authors)} but need {n_authors}).")
    
    chosen_authors = random.sample(eligible_authors, n_authors)
    
    def create_positive_pairs(msgs, n_pairs):
        all_pairs = list(itertools.combinations(msgs, 2))
        if n_pairs > len(all_pairs):
            raise ValueError("Not enough unique pairs can be formed from messages.")
        return random.sample(all_pairs, n_pairs)
    
    training_examples = []
    pos_pairs_mapping = {}
    author_texts_mapping = {}
    
    for author in chosen_authors:
        msgs = author_messages[author]
        pos_pairs = create_positive_pairs(msgs, n_pairs)
        pos_pairs_mapping[author] = pos_pairs
        agg_texts = [msg for pair in pos_pairs for msg in pair]
        author_texts_mapping[author] = agg_texts
        for pair in pos_pairs:
            training_examples.append((pair[0], pair[1], 1, author, author))
    
    for author in chosen_authors:
        own_texts = author_texts_mapping[author]
        other_texts = []
        for other in chosen_authors:
            if other != author:
                for text in author_texts_mapping[other]:
                    other_texts.append((text, other))
        random.shuffle(own_texts)
        random.shuffle(other_texts)
        neg_n = min(n_pairs, len(own_texts), len(other_texts))
        for i in range(neg_n):
            other_text, other_author = other_texts[i]
            training_examples.append((own_texts[i], other_text, 0, author, other_author))
    
    random.shuffle(training_examples)
    
    print("Chosen authors:", chosen_authors)
    print("Number of positive examples:", sum(1 for ex in training_examples if ex[2] == 1))
    print("Number of negative examples:", sum(1 for ex in training_examples if ex[2] == 0))
    print("Sample training examples:", training_examples[:5])
    
    return training_examples, chosen_authors


Chosen authors: ['SaveVideo', 'BubbaSpanks', 'Complete-Trip-9044', 'rBitcoinMod', 'flairtracker']
Number of positive examples: 2000
Number of negative examples: 2000
Sample training examples:
('This transaction has been logged and flairs have been adjusted. Thank you!\n\n  ---\n\n  * u/sarsburner → +73 (Absolute Unit)\n  * u/pimblywimbleton → +2 (Fresh Meat)\n\n  ---', 'This trade is now being followed by the bot. Your flair will update once the other party confirms this trade.\n\nu/kdavants, please reply to the above comment **ONLY AFTER YOUR TRADE IS COMPLETED** and *both* sides have received their end of the transaction. Once you reply, you will both get credit and your flairs will be updated. To confirm this transaction, you must reply with one of the following words: ***positive***, ***confirmed***, or ***confirm***\n\nu/kdavants, if you did **NOT** complete a transaction with the person who tagged you or had a negative experience, please **DO NOT** reply to their comment as this 

Create Testing Data

In [ ]:
training_examples, c = generate_training_data(author_messages, n_authors=5, num_examples=10000)
print(c)
testing_examples, c = generate_training_data(author_messages, n_authors=5, num_examples=1000)
print(c)

In [31]:
authors_testing = set()
for ex in testing_examples:
    authors_testing.add(ex[3])
    authors_testing.add(ex[4])
print(list(authors_testing))

['NeitherVersion2518', 'Cool-Importance6004', 'Harlow_Quinzel', 'coquette_kwason', 'AssistantOnly1987', 'skbast', 'Listige', 'coinbasesupport', 'GothicGolem29', 'qualityvote2', 'No_Departure102', 'UncleGeorge-GPT2', 'SerenCouple', 'HogRideaaaaar', 'SaveVideo', 'Other', 'RicotheWolf24', 'B0tRank', 'AssistanceMods', 'Noobmastter-3000', 'CloudComfortable4346', 'ThePizzaRoolMan', 'CarolusRexhasrisen', 'Scams-ModTeam', 'JuanG_13', 'seattleseahawks2014', 'Peachy_hot_mom', 'avanicks', 'handfulofher', 'Complex-Zucchini-846', 'Vorthex_Celite', 'icpcoc12', 'matthrwmurdock', 'Comprehensive_Cup497', 'Opening-Chapter-9086', 'Rostingu2', 'Alesthar', 'CrazyGoldenSiren', 'reddit_lss_2', 'Leftyperk', 'memes-ModTeam', 'Mrfloydboy', 'julie_4thewin', 'Tsunamicat108', 'Ok-Freedom4496', 'cockobsessedmale', 'hardonenow56', 'Playful-Opening4613', 'WaitingToBeTriggered', 'Bill_VT']


In [ ]:
from transformers import BertModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class SiameseBERT(nn.Module):
    def __init__(self, pretrained_model_name="bert-base-uncased", hidden_size=768, dropout_prob=0.1, n_layers=4):
        """
        Initializes the SiameseBERT network.
        
        Args:
            pretrained_model_name (str): Name of the pretrained BERT model.
            hidden_size (int): Hidden size of the BERT model.
            dropout_prob (float): Dropout probability.
            n_layers (int): Number of last layers to use for layer-wise representations.
        """
        super(SiameseBERT, self).__init__()
        self.n_layers = n_layers
        
        # Load BERT and enable output of hidden states for layer-wise representations
        self.bert = BertModel.from_pretrained(pretrained_model_name, output_hidden_states=True)
        self.dropout = nn.Dropout(dropout_prob)
        
        # Learnable weights for combining the last n_layers representations
        self.layer_weights = nn.Parameter(torch.ones(n_layers) / n_layers)
        # Learnable vector for attention pooling across tokens to capture grammatical structure
        self.attn_vector = nn.Parameter(torch.randn(hidden_size))
        
        # Fully connected block: main branch
        self.fc_layers = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU()
        )
        # Residual branch to map the original embedding to the reduced dimension
        self.fc_residual = nn.Linear(hidden_size, hidden_size // 2)
        
        # Layer normalization after adding the residual connection
        self.layer_norm = nn.LayerNorm(hidden_size // 2)

    def attention_pool(self, tokens, mask):
        """
        Applies attention pooling over token embeddings to capture grammatical structure.
        
        Args:
            tokens (torch.Tensor): Token embeddings of shape (batch_size, seq_len, hidden_size).
            mask (torch.Tensor): Attention mask of shape (batch_size, seq_len).
            
        Returns:
            torch.Tensor: Pooled embedding of shape (batch_size, hidden_size).
        """
        # Compute attention scores using a learnable vector
        scores = torch.matmul(tokens, self.attn_vector)  # shape: (batch_size, seq_len)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # shape: (batch_size, seq_len, 1)
        pooled = torch.sum(tokens * attn_weights, dim=1)  # shape: (batch_size, hidden_size)
        return pooled

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2):
        """
        Forward pass for the Siamese network.
        
        Args:
            input_ids1 (torch.Tensor): Input IDs for the first text, shape (batch_size, seq_len).
            attention_mask1 (torch.Tensor): Attention mask for the first text.
            input_ids2 (torch.Tensor): Input IDs for the second text.
            attention_mask2 (torch.Tensor): Attention mask for the second text.
            
        Returns:
            prob (torch.Tensor): Predicted probability (same author) of shape (batch_size, 1).
            embed1 (torch.Tensor): Refined embedding for the first text.
            embed2 (torch.Tensor): Refined embedding for the second text.
        """
        # Batch processing: concatenate inputs along the batch dimension
        input_ids = torch.cat([input_ids1, input_ids2], dim=0)  # Shape: (2*batch_size, seq_len)
        attention_mask = torch.cat([attention_mask1, attention_mask2], dim=0)
        
        # Get BERT outputs with hidden states
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states  # Tuple of layers: (layer0, layer1, ..., layer_n)
        
        # Instead of just using the [CLS] token, apply attention pooling on each layer
        pooled_embeddings = []
        for i in range(1, self.n_layers + 1):
            tokens = hidden_states[-i]  # shape: (batch, seq_len, hidden_size)
            pooled = self.attention_pool(tokens, attention_mask)
            pooled_embeddings.append(pooled)
        weighted_pooled = sum(w * emb for w, emb in zip(self.layer_weights, pooled_embeddings))
        weighted_pooled = self.dropout(weighted_pooled)
        
        # Fully connected transformation with residual connection and normalization
        refined = self.fc_layers(weighted_pooled) + self.fc_residual(weighted_pooled)
        refined = self.layer_norm(refined)
        
        # Split back into two halves for the two inputs
        batch_size = input_ids1.size(0)
        embed1 = refined[:batch_size]
        embed2 = refined[batch_size:]
        
        return embed1, embed2

def contrastive_loss(embedding1, embedding2, label, margin=0.5):
    """
    Computes the contrastive loss for metric learning.
    
    Args:
        embedding1 (torch.Tensor): Embeddings for the first input.
        embedding2 (torch.Tensor): Embeddings for the second input.
        label (torch.Tensor): 1 if same author, 0 otherwise.
        margin (float): Margin for the contrastive loss.
    
    Returns:
        torch.Tensor: Mean contrastive loss.
    """
     # Normalize the embeddings to unit vectors
    output1_norm = F.normalize(embedding1, p=2, dim=1)
    output2_norm = F.normalize(embedding2, p=2, dim=1)
    
    # Compute cosine similarity between each pair in the batch
    cosine_sim = F.cosine_similarity(output1_norm, output2_norm, dim=1)
    
    # For similar pairs (label==1), we want the similarity to be close to 1.
    # Loss for similar pairs: 1 - cosine similarity.
    loss_similar = (1 - cosine_sim) * label
    
    # For dissimilar pairs (label==0), if the cosine similarity is greater than the margin,
    # we want to push it down. Otherwise, no loss is incurred.
    loss_dissimilar = F.relu(cosine_sim - margin) * (1 - label)
    
    # Combine the losses and take the mean over the batch.
    loss = torch.mean(loss_similar + loss_dissimilar)
    return loss



In [ ]:
from sklearn.model_selection import train_test_split
# Initialize tokenizer (if not already defined)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Extract texts and labels from training_examples 
texts1 = [ex[0] for ex in training_examples]
texts2 = [ex[1] for ex in training_examples]
labels = [ex[2] for ex in training_examples]

max_length = 512
encoded_inputs1 = tokenizer(texts1, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
encoded_inputs2 = tokenizer(texts2, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")

# Convert labels to a tensor
labels = torch.tensor(labels, dtype=torch.float)

# Pack tokenized data into a dictionary
# Split data into train and validation sets
train_ids1, val_ids1, train_masks1, val_masks1, train_ids2, val_ids2, train_masks2, val_masks2, train_labels, val_labels, train_author1, val_author1, train_author2, val_author2 = train_test_split(
    encoded_inputs1["input_ids"],
    encoded_inputs1["attention_mask"],
    encoded_inputs2["input_ids"],
    encoded_inputs2["attention_mask"],
    labels,
    [ex[3] for ex in training_examples],
    [ex[4] for ex in training_examples],
    test_size=0.2,
    random_state=42
)

training_data = {
    "input_ids1": train_ids1,
    "attention_mask1": train_masks1,
    "input_ids2": train_ids2,
    "attention_mask2": train_masks2,
    "labels": train_labels,
    "author1": train_author1,
    "author2": train_author2
}

train_validation_data = {
    "input_ids1": val_ids1,
    "attention_mask1": val_masks1,
    "input_ids2": val_ids2,
    "attention_mask2": val_masks2,
    "labels": val_labels,
    "author1": val_author1,
    "author2": val_author2
}

print("Tokenization  of Training complete. Example input_ids1 for first example:")
print(training_data["input_ids1"][0])

#do the same on the testing examples
texts1 = [ex[0] for ex in testing_examples]
texts2 = [ex[1] for ex in testing_examples]
labels = [ex[2] for ex in testing_examples]

max_length = 512
encoded_inputs1 = tokenizer(texts1, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
encoded_inputs2 = tokenizer(texts2, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")

# Convert labels to a tensor
labels = torch.tensor(labels, dtype=torch.float)

# Pack tokenized data into a dictionary
testing_data = {
    "input_ids1": encoded_inputs1["input_ids"],
    "attention_mask1": encoded_inputs1["attention_mask"],
    "input_ids2": encoded_inputs2["input_ids"],
    "attention_mask2": encoded_inputs2["attention_mask"],
    "labels": labels,
    "author1": [ex[3] for ex in testing_examples], 
    "author2": [ex[4] for ex in testing_examples]
}

print("Tokenization of Testing complete. Example input_ids1 for first example:")
print(testing_data["input_ids1"][0])

Tokenization  of Training complete. Example input_ids1 for first example:
tensor([  101,  3504,  2200, 11937, 21756, 11561,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     

In [24]:
# Combine the two author lists and then print the unique authors
unique_authors = set(testing_data['author1'] + testing_data['author2'])
print(unique_authors)

{'Parklifeee', 'AssistanceMods', 'awabg-kun', 'Noobmastter-3000', 'SnooOpinions5223', 'timmyblur', 'Radkingeli995', 'alwaysfatigued8787'}


In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib

def eval_model(model,device, testing_data, name = ""):
    
    # Visualize validation embeddings after the epoch
    model.eval()
    all_embeddings = []
    embedding_pairs = []
    all_authors = []
    labels = []
    val_indexs = range(len(testing_data["input_ids1"]))
    
    unique_authors = set(testing_data['author1'] + testing_data['author2'])
    
    with torch.no_grad():
        for idx in tqdm(val_indexs, desc="Processing validation indices"):
            sample_ids1 = testing_data["input_ids1"][idx].unsqueeze(0).to(device)
            sample_mask1 = testing_data["attention_mask1"][idx].unsqueeze(0).to(device)
            sample_ids2 = testing_data["input_ids2"][idx].unsqueeze(0).to(device)
            sample_mask2 = testing_data["attention_mask2"][idx].unsqueeze(0).to(device)
            
            emb1, emb2 = model(sample_ids1, sample_mask1, sample_ids2, sample_mask2)
            all_embeddings.append(emb1.cpu())
            all_authors.append(testing_data['author1'][idx])
            all_embeddings.append(emb2.cpu())
            all_authors.append(testing_data['author2'][idx])
            
            embedding_pairs.append((emb1.cpu(), emb2.cpu()))
            labels.append(testing_data['labels'][idx])
    
    embeddings_np = torch.cat(all_embeddings, dim=0).numpy()
    le = LabelEncoder()
    encoded_authors = le.fit_transform(np.array(all_authors))

    # SVM 1: Individual Embeddings
    X_train, X_test, y_train, y_test = train_test_split(embeddings_np, encoded_authors,
                                                        test_size=0.2, random_state=42)
    svm_ind = SVC(kernel='rbf')
    svm_ind.fit(X_train, y_train)
    preds_ind = svm_ind.predict(X_test)
    acc_ind = accuracy_score(y_test, preds_ind) * 100

    # SVM 2: Pair-Difference SVM
    diffs = []
    for emb1, emb2 in embedding_pairs:
        diff = np.array([F.cosine_similarity(emb1, emb2).item()])
        diffs.append(diff)
    diffs = np.array(diffs)
    labels_arr = np.array(labels)
    
    X_train_pair, X_test_pair, y_train_pair, y_test_pair = train_test_split(diffs, labels_arr,
                                                                            test_size=0.2, random_state=42)
    svm_pair = SVC(kernel='rbf')
    svm_pair.fit(X_train_pair, y_train_pair)
    preds_pair = svm_pair.predict(X_test_pair)
    acc_pair = accuracy_score(y_test_pair, preds_pair) * 100

    # SVM 3: Author Verification from predicted labels
    _, test_pairs, _, y_pair_test = train_test_split(embedding_pairs, labels,
                                                        test_size=0.2, random_state=42)
    author_pred_labels = []
    for emb1, emb2 in test_pairs:
        pred1 = svm_ind.predict(emb1.numpy())
        pred2 = svm_ind.predict(emb2.numpy())
        author_pred_labels.append(1 if pred1[0] == pred2[0] else 0)
    acc_auth = accuracy_score(np.array(y_pair_test), np.array(author_pred_labels)) * 100

    # Optional: Author verification by distance of decision_function
    author_pred_logits = []
    for emb1, emb2 in test_pairs:
        logits1 = svm_ind.decision_function(emb1.numpy())
        logits2 = svm_ind.decision_function(emb2.numpy())
        distance = np.linalg.norm(logits1 - logits2)
        author_pred_logits.append(1 if distance < 0.5 else 0)
    acc_auth_logits = accuracy_score(np.array(y_pair_test), np.array(author_pred_logits)) * 100
    
    # PCA Visualization
    all_embeddings = torch.cat(all_embeddings, dim=0).numpy()
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(all_embeddings)

    plt.figure(figsize=(8, 6))
    filtered_authors = [a for a in all_authors if a.lower() != 'other']
    filtered_indices = [j for j, a in enumerate(all_authors) if a.lower() != 'other']
    embeddings_2d_filtered = embeddings_2d[filtered_indices]
    unique_authors = list(set(filtered_authors))
    colors = matplotlib.cm.get_cmap("tab20", lut=len(unique_authors))
    for i, author in enumerate(unique_authors):
        inds = [j for j, a in enumerate(filtered_authors) if a == author]
        plt.scatter(embeddings_2d_filtered[inds, 0], embeddings_2d_filtered[inds, 1],
                    color=colors(i), label=author, alpha=0.7)

    accuracy_text = (
        f"(Classification): {acc_ind:.2f}%\n"
        f"Embedding (Verification): {acc_pair:.2f}%\n"
        f"Predicted Author (Verification): {acc_auth:.2f}%\n"
        f"Logits (Verification): {acc_auth_logits:.2f}%"
    )
    plt.text(1.05, 0.5, accuracy_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='center', bbox=dict(facecolor='white', alpha=0.5))

    plt.title(f'{name} Embeddings')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()


In [ ]:
import gc
import numpy as np


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
debug = False

# Prepare dataset using tensors 
training_dataset = TensorDataset(
    training_data["input_ids1"],
    training_data["attention_mask1"],
    training_data["input_ids2"],
    training_data["attention_mask2"],
    training_data["labels"]
)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the updated SiameseBERT model and move it to device
model = SiameseBERT().to(device)

# Use AdamW optimizer with a low learning rate
optimizer = AdamW(model.parameters(), lr=2e-5)

# Define training parameters
n_epochs = 3
batch_size = 8


for param in model.bert.encoder.parameters():
    param.requires_grad = True

# Re-initialize the top layers (9–11) with random values
for idx in [8, 9, 10, 11]:
    for param in model.bert.encoder.layer[idx].parameters():
        torch.nn.init.normal_(param.data, mean=0.0, std=0.02)

# Calculate total training steps for scheduler
total_steps = (len(training_dataset) // batch_size) * n_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=int(0.1 * total_steps), 
                                            num_training_steps=total_steps)

train_subset = training_dataset
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)

# Train the model
for epoch in range(1, n_epochs + 1):
    model.train()
    train_loss = 0
    total = 0
    batch_losses = []  # record loss for each batch
    
    # Use enumerate to get the batch index in the loop
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}"), start=1):
        
        # Unpack batch and move to device
        input_ids1, mask1, input_ids2, mask2, labels = batch
        # Move tensors to device
        input_ids1 = input_ids1.to(device)
        mask1 = mask1.to(device)
        input_ids2 = input_ids2.to(device)
        mask2 = mask2.to(device)
        labels = labels.to(device).float()
        
        optimizer.zero_grad()
        emb1, emb2 = model(input_ids1, mask1, input_ids2, mask2)

        # Compute contrastive loss
        loss = contrastive_loss(emb1, emb2, labels, margin=1.0)
        
        loss.backward()
        optimizer.step()
        scheduler.step()  # update learning rate scheduler
        
        train_loss += loss.item() * labels.size(0)
        total += labels.size(0)
        batch_losses.append(loss.item())
        
        if i == 1 or (i % max(1, int(0.2 * len(train_loader))) == 0):
            model.eval()
            
            eval_model(model, device, testing_data, f"Batch{i} Testing")
            eval_model(model, device, train_validation_data, f"Batch{i} Training")
            
            model.train()
    
    avg_loss = train_loss / total
    print(f"Epoch {epoch} - Loss: {avg_loss:.4f} ")
    
    print(f"Epoch {epoch} Testing Results")
    eval_model(model, device, testing_data, f"Epoch {epoch} Testing")
    
    print(f"Epoch {epoch} Training Results")
    eval_model(model, device, train_validation_data, f'Epoch {epoch} Training')
    
    # Smooth out the loss graph using a moving average filter
    smoothing_window = 5  # Adjust the window size as needed
    smooth_losses = np.convolve(batch_losses, np.ones(smoothing_window)/smoothing_window, mode='valid')
    
    plt.figure(figsize=(8, 4))
    plt.plot(range(smoothing_window, len(batch_losses) + 1), smooth_losses, marker='o')
    plt.title(f'Epoch {epoch}: Smoothed Batch vs Loss')
    plt.xlabel('Batch Number')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.show()
